# Notebook 2: Demand Forecasting Model
**Goal:** predict hotel **occupancy (demand)** for each day.

We train 3 models and compare them with a simple baseline:
| Model | Why |
|---|---|
| Naive baseline | "Same as 7 days ago" - a model must beat this |
| Linear Regression | Simple starting model |
| Random Forest | Main ML model |
| XGBoost | Main ML model |

**Important:** we do NOT use competitor price or the target price as inputs. Demand should be predicted from tourism data only.

## Step 1: Load the clean data

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pricing_project'
except ImportError:
    BASE = os.environ.get('PROJECT_DIR', '.')

MODELS = f'{BASE}/models'; RESULTS = f'{BASE}/results'
os.makedirs(MODELS, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

df = pd.read_csv(f'{BASE}/data/clean/main_clean.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
print(df.shape)

## Step 2: Choose inputs and split by time (80% train / 20% test)
We split by **date**, not randomly: the model learns from the past (first 80%) and is tested on the future (last 20%). This is more honest for time data.

In [ ]:
TARGET = 'Historical_Occupancy_Rate'
FEATURES = ['SLTDA_Tourist_Arrivals', 'Booking_Lead_Time', 'Seasonality_Index_Norm', 'Season_Code',
            'Day_of_Week', 'Is_Weekend', 'Month', 'Demand_Surge_Flag',
            'Arrivals_Lag7', 'Arrivals_Roll7', 'Occupancy_Lag7']

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print('Train:', len(train), 'days ->', train.Date.min().date(), 'to', train.Date.max().date())
print('Test :', len(test),  'days ->', test.Date.min().date(),  'to', test.Date.max().date())

## Step 3: Helper function for the 3 error measures

In [ ]:
def scores(y_true, y_pred):
    return {'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAE':  mean_absolute_error(y_true, y_pred),
            'MAPE_%': mean_absolute_percentage_error(y_true, y_pred) * 100}

results, preds = {}, {}

# Baseline: same as 7 days ago
preds['Naive (7 days ago)'] = test['Occupancy_Lag7'].values
results['Naive (7 days ago)'] = scores(y_test, preds['Naive (7 days ago)'])

## Step 4: Linear Regression

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
preds['Linear Regression'] = lr.predict(X_test)
results['Linear Regression'] = scores(y_test, preds['Linear Regression'])
results['Linear Regression']

## Step 5: Random Forest (with simple tuning)
Tuning uses `TimeSeriesSplit`, so the validation folds are also in time order.

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [200, 300, 500], 'max_depth': [4, 6, 8, None],
     'min_samples_leaf': [1, 3, 5, 10], 'max_features': [0.5, 0.8, 1.0]},
    n_iter=15, cv=tscv, scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)
rf = rf_search.best_estimator_
print('Best Random Forest settings:', rf_search.best_params_)

preds['Random Forest'] = rf.predict(X_test)
results['Random Forest'] = scores(y_test, preds['Random Forest'])
results['Random Forest']

## Step 6: XGBoost (with simple tuning)

In [ ]:
xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    {'n_estimators': [200, 300, 500], 'learning_rate': [0.01, 0.03, 0.05, 0.1],
     'max_depth': [2, 3, 4, 5], 'subsample': [0.7, 0.9, 1.0], 'colsample_bytree': [0.7, 0.9, 1.0]},
    n_iter=20, cv=tscv, scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1)
xgb_search.fit(X_train, y_train)
xgb = xgb_search.best_estimator_
print('Best XGBoost settings:', xgb_search.best_params_)

preds['XGBoost'] = xgb.predict(X_test)
results['XGBoost'] = scores(y_test, preds['XGBoost'])
results['XGBoost']

## Step 7: Compare all models

In [ ]:
table = pd.DataFrame(results).T.round(2).sort_values('RMSE')
table.to_csv(f'{RESULTS}/model_comparison.csv')
table

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(test.Date, y_test.values, label='Actual', color='black', linewidth=2)
for name in ['Linear Regression', 'Random Forest', 'XGBoost']:
    ax.plot(test.Date, preds[name], label=name, alpha=0.8)
ax.set_title('Actual vs predicted occupancy (test period)'); ax.set_ylabel('Occupancy %'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RESULTS}/actual_vs_predicted.png', dpi=150); plt.show()

## Step 8: Which inputs matter most? (XGBoost)

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind='barh', figsize=(8, 5), title='XGBoost feature importance')
plt.tight_layout(); plt.savefig(f'{RESULTS}/feature_importance.png', dpi=150); plt.show()

## Step 9: Save the models for the next steps

In [ ]:
joblib.dump(lr,  f'{MODELS}/linear_regression.pkl')
joblib.dump(rf,  f'{MODELS}/random_forest.pkl')
joblib.dump(xgb, f'{MODELS}/xgboost.pkl')
joblib.dump(FEATURES, f'{MODELS}/features.pkl')
print('Saved 3 models and the feature list in', MODELS)

## Done
Write the results table in your thesis (Chapter on Evaluation). Remember to explain:
- Time-based 80/20 split (test = the last 20% of days)
- The naive baseline and why the ML models must beat it
- If Linear Regression is as good as or better than the tree models, say so honestly: it means the data is mostly simple and linear.

**Next: Notebook 3 - Pricing engine, revenue simulation and SHAP explanations.**